<a href="https://colab.research.google.com/github/AshokGit544/Enterprise-Model-Quantization-Platform1/blob/main/Enterprise_Model_Quantization_Platform1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pandas numpy scikit-learn

In [2]:
from pathlib import Path
from datetime import datetime
import random
import json
import re
import time

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [3]:
BASE_PATH = Path('/content/enterprise_model_quantization_platform')
RAW_PATH = BASE_PATH / 'data' / 'raw_documents'
PROCESSED_PATH = BASE_PATH / 'data' / 'processed_documents'
FEATURE_PATH = BASE_PATH / 'data' / 'features'
OUTPUT_PATH = BASE_PATH / 'outputs'
CONFIG_PATH = BASE_PATH / 'config'

for p in [RAW_PATH, PROCESSED_PATH, FEATURE_PATH, OUTPUT_PATH, CONFIG_PATH]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(42)
np.random.seed(42)

print('Project folders created successfully.')
print(BASE_PATH)

Project folders created successfully.
/content/enterprise_model_quantization_platform


In [4]:
project_schema = {
    'document_types': [
        'quality_report',
        'supplier_invoice',
        'maintenance_record',
        'engineering_change_notice'
    ],
    'deployment_goal': 'reduce_model_size_and_improve_inference_efficiency',
    'quantization_concept': {
        'original_precision': 'float64_or_float32_style_weights',
        'quantized_precision': 'int8_style_weights',
        'business_goal': 'improve_deployability_runtime_efficiency_and_cost_conscious_scaling'
    }
}

with open(CONFIG_PATH / 'project_schema.json', 'w') as f:
    json.dump(project_schema, f, indent=2)

print(json.dumps(project_schema, indent=2))

{
  "document_types": [
    "quality_report",
    "supplier_invoice",
    "maintenance_record",
    "engineering_change_notice"
  ],
  "deployment_goal": "reduce_model_size_and_improve_inference_efficiency",
  "quantization_concept": {
    "original_precision": "float64_or_float32_style_weights",
    "quantized_precision": "int8_style_weights",
    "business_goal": "improve_deployability_runtime_efficiency_and_cost_conscious_scaling"
  }
}


In [5]:
documents = []

supplier_names = ['Magna', 'Bosch', 'Denso', 'Lear', 'Aptiv']
plants = ['PLT100', 'PLT200', 'PLT300', 'PLT400']
issue_types = ['weld defect', 'paint inconsistency', 'sensor failure', 'material shortage', 'assembly variance']
equipment_ids = ['EQP1001', 'EQP1002', 'EQP1003', 'EQP1004']

for i in range(1, 1001):
    doc_type = random.choice(project_schema['document_types'])
    plant = random.choice(plants)
    supplier = random.choice(supplier_names)
    issue = random.choice(issue_types)
    equipment = random.choice(equipment_ids)
    doc_date = f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}"
    amount = max(1000.0, round(float(np.random.normal(12500, 3500)), 2))
    change_order_id = f"ECO-{random.randint(10000,99999)}"

    if doc_type == 'quality_report':
        text = (
            f"Document ID DOC{i:05d}. Plant {plant}. Quality inspection found {issue} on equipment {equipment}. "
            f"Document date {doc_date}. Supplier reference {supplier}. "
            f"The report explains defect review, containment action, severity review, and production quality impact."
        )
    elif doc_type == 'supplier_invoice':
        text = (
            f"Document ID DOC{i:05d}. Supplier invoice from {supplier} for plant {plant}. "
            f"Invoice amount ${amount}. Document date {doc_date}. Material issue related to {issue}. "
            f"The invoice includes purchasing notes, operational dependency, and cost review information."
        )
    elif doc_type == 'maintenance_record':
        text = (
            f"Document ID DOC{i:05d}. Maintenance record for equipment {equipment} at plant {plant}. "
            f"Reported issue {issue}. Service date {doc_date}. Vendor {supplier}. "
            f"The note explains service action, downtime effect, equipment status, and recommended follow-up."
        )
    else:
        text = (
            f"Document ID DOC{i:05d}. Engineering change notice {change_order_id} for plant {plant}. "
            f"Change related to {issue}. Effective date {doc_date}. Supplier {supplier}. "
            f"The notice explains implementation details, engineering review, and downstream manufacturing impact."
        )

    documents.append({
        'document_id': f'DOC{i:05d}',
        'document_type': doc_type,
        'document_text': text
    })

documents_df = pd.DataFrame(documents)
documents_df.to_csv(RAW_PATH / 'enterprise_documents.csv', index=False)

print(documents_df.head())
print('Total documents:', len(documents_df))

  document_id   document_type  \
0    DOC00001  quality_report   
1    DOC00002  quality_report   
2    DOC00003  quality_report   
3    DOC00004  quality_report   
4    DOC00005  quality_report   

                                       document_text  
0  Document ID DOC00001. Plant PLT100. Quality in...  
1  Document ID DOC00002. Plant PLT400. Quality in...  
2  Document ID DOC00003. Plant PLT200. Quality in...  
3  Document ID DOC00004. Plant PLT200. Quality in...  
4  Document ID DOC00005. Plant PLT100. Quality in...  
Total documents: 1000


In [6]:
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

documents_df = pd.read_csv(RAW_PATH / 'enterprise_documents.csv')
documents_df['normalized_text'] = documents_df['document_text'].apply(normalize_text)

documents_df.to_csv(PROCESSED_PATH / 'processed_documents.csv', index=False)

print(documents_df.head())

  document_id   document_type  \
0    DOC00001  quality_report   
1    DOC00002  quality_report   
2    DOC00003  quality_report   
3    DOC00004  quality_report   
4    DOC00005  quality_report   

                                       document_text  \
0  Document ID DOC00001. Plant PLT100. Quality in...   
1  Document ID DOC00002. Plant PLT400. Quality in...   
2  Document ID DOC00003. Plant PLT200. Quality in...   
3  Document ID DOC00004. Plant PLT200. Quality in...   
4  Document ID DOC00005. Plant PLT100. Quality in...   

                                     normalized_text  
0  document id doc00001. plant plt100. quality in...  
1  document id doc00002. plant plt400. quality in...  
2  document id doc00003. plant plt200. quality in...  
3  document id doc00004. plant plt200. quality in...  
4  document id doc00005. plant plt100. quality in...  


In [7]:
X = documents_df['normalized_text']
y = documents_df['document_type']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(stop_words='english', max_features=500)
X_train_vec = vectorizer.fit_transform(X_train).toarray()
X_test_vec = vectorizer.transform(X_test).toarray()

print('Train shape:', X_train_vec.shape)
print('Test shape:', X_test_vec.shape)

Train shape: (800, 500)
Test shape: (200, 500)


In [8]:
base_clf = LogisticRegression(max_iter=2000, random_state=42)
base_clf.fit(X_train_vec, y_train)

base_preds = base_clf.predict(X_test_vec)
base_accuracy = accuracy_score(y_test, base_preds)

print('Base model accuracy:', round(float(base_accuracy), 4))
print(classification_report(y_test, base_preds))

Base model accuracy: 1.0
                           precision    recall  f1-score   support

engineering_change_notice       1.00      1.00      1.00        52
       maintenance_record       1.00      1.00      1.00        48
           quality_report       1.00      1.00      1.00        50
         supplier_invoice       1.00      1.00      1.00        50

                 accuracy                           1.00       200
                macro avg       1.00      1.00      1.00       200
             weighted avg       1.00      1.00      1.00       200



In [9]:
coef_bytes_original = base_clf.coef_.nbytes
intercept_bytes_original = base_clf.intercept_.nbytes
total_bytes_original = coef_bytes_original + intercept_bytes_original

original_memory_summary = {
    'coef_bytes_original': int(coef_bytes_original),
    'intercept_bytes_original': int(intercept_bytes_original),
    'total_bytes_original': int(total_bytes_original)
}

with open(OUTPUT_PATH / 'original_memory_summary.json', 'w') as f:
    json.dump(original_memory_summary, f, indent=2)

print(json.dumps(original_memory_summary, indent=2))

{
  "coef_bytes_original": 16000,
  "intercept_bytes_original": 32,
  "total_bytes_original": 16032
}


In [10]:
start_time = time.perf_counter()

for _ in range(200):
    _ = base_clf.predict(X_test_vec)

end_time = time.perf_counter()
base_inference_time = end_time - start_time

print('Base model repeated inference time:', round(float(base_inference_time), 6), 'seconds')

Base model repeated inference time: 0.15004 seconds


In [15]:
def quantize_array_to_int8(arr):
    arr = np.asarray(arr, dtype=np.float32)
    max_abs = np.max(np.abs(arr))

    if max_abs == 0:
        scale = 1.0
    else:
        scale = max_abs / 127.0

    quantized = np.round(arr / scale).astype(np.int8)
    return quantized, scale

def dequantize_array_from_int8(quantized_arr, scale):
    return quantized_arr.astype(np.float32) * scale

In [16]:
quantized_coef, coef_scale = quantize_array_to_int8(base_clf.coef_)
quantized_intercept, intercept_scale = quantize_array_to_int8(base_clf.intercept_)

print("Quantized coef shape:", quantized_coef.shape)
print("Quantized intercept shape:", quantized_intercept.shape)
print("Coefficient scale:", coef_scale)
print("Intercept scale:", intercept_scale)

Quantized coef shape: (4, 500)
Quantized intercept shape: (4,)
Coefficient scale: 0.026002143
Intercept scale: 0.002342459


In [17]:
dequantized_coef = dequantize_array_from_int8(quantized_coef, coef_scale)
dequantized_intercept = dequantize_array_from_int8(quantized_intercept, intercept_scale)

print("Dequantized coef shape:", dequantized_coef.shape)
print("Dequantized intercept shape:", dequantized_intercept.shape)

Dequantized coef shape: (4, 500)
Dequantized intercept shape: (4,)


In [18]:
class_labels = base_clf.classes_

def quantized_predict(X, coef_matrix, intercept_vector, class_labels):
    scores = np.dot(X, coef_matrix.T) + intercept_vector
    pred_indices = np.argmax(scores, axis=1)
    return class_labels[pred_indices]

In [19]:
quantized_preds = quantized_predict(
    X_test_vec,
    dequantized_coef,
    dequantized_intercept,
    class_labels
)

print(quantized_preds[:10])

['supplier_invoice' 'maintenance_record' 'supplier_invoice'
 'supplier_invoice' 'supplier_invoice' 'quality_report'
 'maintenance_record' 'quality_report' 'supplier_invoice'
 'supplier_invoice']


In [20]:
quantized_accuracy = accuracy_score(y_test, quantized_preds)

print("Quantized model accuracy:", round(float(quantized_accuracy), 4))
print(classification_report(y_test, quantized_preds))

Quantized model accuracy: 1.0
                           precision    recall  f1-score   support

engineering_change_notice       1.00      1.00      1.00        52
       maintenance_record       1.00      1.00      1.00        48
           quality_report       1.00      1.00      1.00        50
         supplier_invoice       1.00      1.00      1.00        50

                 accuracy                           1.00       200
                macro avg       1.00      1.00      1.00       200
             weighted avg       1.00      1.00      1.00       200



In [21]:
coef_bytes_original = base_clf.coef_.nbytes
intercept_bytes_original = base_clf.intercept_.nbytes
total_bytes_original = coef_bytes_original + intercept_bytes_original

coef_bytes_quantized = quantized_coef.nbytes
intercept_bytes_quantized = quantized_intercept.nbytes
total_bytes_quantized = coef_bytes_quantized + intercept_bytes_quantized

print("Original model bytes:", total_bytes_original)
print("Quantized model bytes:", total_bytes_quantized)
print("Compression ratio:", round(total_bytes_original / total_bytes_quantized, 4))

Original model bytes: 16032
Quantized model bytes: 2004
Compression ratio: 8.0


In [22]:
import time

start_time = time.perf_counter()

for _ in range(200):
    _ = quantized_predict(
        X_test_vec,
        dequantized_coef,
        dequantized_intercept,
        class_labels
    )

end_time = time.perf_counter()
quantized_inference_time = end_time - start_time

print("Quantized inference time:", round(float(quantized_inference_time), 6), "seconds")

Quantized inference time: 0.054651 seconds


In [23]:
comparison_df = pd.DataFrame({
    'model_version': ['original_model', 'quantized_model'],
    'accuracy': [round(float(base_accuracy), 4), round(float(quantized_accuracy), 4)],
    'memory_bytes': [int(total_bytes_original), int(total_bytes_quantized)]
})

print(comparison_df)

     model_version  accuracy  memory_bytes
0   original_model       1.0         16032
1  quantized_model       1.0          2004


In [24]:
compression_ratio = total_bytes_original / total_bytes_quantized
print("Compression ratio:", round(compression_ratio, 2))

Compression ratio: 8.0
